In [2]:
import pandas as pd

from pathlib import Path
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [3]:
PROJECT_DIR = Path.cwd().parent

CURATED_DIR = (
    PROJECT_DIR
    / "data"
    / "curated"
)

policy_360 = pd.read_csv(
    CURATED_DIR / "policy_360.csv",
    low_memory=False
)

print(policy_360.shape)

(150000, 88)


In [4]:
print(
    policy_360["POLICY_ID"].nunique()
)

print(
    policy_360["POLICY_ID"]
    .duplicated()
    .sum()
)

150000
0


In [5]:
date_cols = [
    "POLICY_START_DATE",
    "POLICY_END_DATE",
    "CUSTOMER_SINCE",
    "LAST_CLAIM_DATE"
]

for col in date_cols:
    if col in policy_360.columns:
        policy_360[col] = pd.to_datetime(policy_360[col], errors="coerce")

In [6]:
SERVER_NAME = r"."
DATABASE_NAME = "InsuranceAnalyticsDB"

In [7]:
connection_string = quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)
print('conn susscesful ')

conn susscesful 


In [8]:
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_string}",
    fast_executemany=True
)

In [9]:
test_query = """
SELECT DB_NAME() AS CURRENT_DATABASE
"""

pd.read_sql(
    test_query,
    engine
)

,CURRENT_DATABASE
0,InsuranceAnalyticsDB


In [10]:
policy_360.to_sql(
    name="policy_360",
    con=engine,
    schema="analytics",
    if_exists="replace",
    index=False,
    chunksize=1000
)

print(
    "Policy 360 uploaded successfully."
)

Policy 360 uploaded successfully.


In [11]:
query = """
SELECT
    POLICY_TYPE,
    COUNT(*) AS POLICY_COUNT,
    AVG(ANNUAL_PREMIUM) AS AVG_ANNUAL_PREMIUM
FROM analytics.policy_360
GROUP BY POLICY_TYPE
ORDER BY POLICY_COUNT DESC
"""

product_sql_df = pd.read_sql(query, engine)

product_sql_df

,POLICY_TYPE,POLICY_COUNT,AVG_ANNUAL_PREMIUM
0,Health,36123,19143
1,Motor,31768,14892
2,Term Life,23933,17041
3,Commercial,14892,47712
4,Home,12031,11681
5,Personal Accident,11989,5847
6,Whole Life,10302,34195
7,Travel,8962,3720
